# $Bx$ component as Ito's process 
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import numpy as np
from magfield.visual.approximation import harmonic_approximation
from magfield.visual.plot import long_plot
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
import pickle
from tqdm.notebook import tqdm


# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

## Calculate $a(t), \space b(t)$

In [ ]:
# Load up gaussian mixture model for component Bx
with open("data/full_dBx_5000_3.pkl", "rb") as f:
    gmm = pickle.load(f)

Parameters of Ito's equation are calculated in the next way:

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
b(t) = \sum_{k=1}^{K}{p_k b_k},
$$
where $K$ is a number of mixture components

In [ ]:
p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]

coef_a = np.sum(p * a, axis=0)
coef_b = np.sum(p * b, axis=0)

# Correlation plots

In [ ]:
correlation_window = (gmm["window"]["size"], gmm["window"]["step"])
kernel_size = 60 * 4  # For smoothing plots

#### Correlation $a(t)$, $b(t)=\sum_{j=1}^Kp_jb_j$

In [ ]:
correlation = []
for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
title = rf"""$\text{{Correlation between }} a(t), b(t)=
    \sum_{{j=1}}^{gmm["num_comp"]}p_jb_j \text{{ on window size }}
    {correlation_window[0]} \text{{ minutes}}$
    """
cp = long_plot(correlation, title, dates=gmm["dates"], n=12)
cp.show()
nc = gmm["num_comp"]
cp.write_image(f"/tmp/dBx_corr{nc}_{correlation_window[0]}.png")

#### Correlation $a(t)$, $b^2(t)$

In [ ]:
# correlation2 = []
# for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
#     sub_a = coef_a[i : correlation_window[0] + i]
#     sub_b = (coef_b**2)[i : correlation_window[0] + i]
#     correlation2.append(np.corrcoef(sub_a, sub_b)[0, 1])

In [ ]:
# title = (
#     rf"""$\text{{Correlation between }} a(t), b^2(t)=
#     \left(\sum_{{j=1}}^{gmm["num_comp"]}p_jb_j\right)^2 \text{{ on window size }}
#     {correlation_window[0]} \text{{ minutes}}$""")
# cp = long_plot(correlation2, title)
# cp.show()
# nc = gmm["num_comp"]
# cp.write_image(f"/tmp/dBx_corr_sqr_{nc}_{correlation_window[0]}.png")

# Analysis of $a(t)$ and $b(t)$ relation

## Trigonometric approximation

In [ ]:
hn = 16
fig, resids, *_ = harmonic_approximation(
    data=correlation,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    title="Bx - correlation a(t), b(t)",
)
del _
fig.show()
# fig.write_image(f"/tmp/dBx_harmonic_approx_{hn}_{correlation_window[0]}.png")

In [ ]:
from statsmodels.graphics.gofplots import qqplot

qqplot_data = qqplot(resids, line="s").gca().lines
# fig = go.Figure()

# fig.add_trace({
#     'type': 'scatter',
#     'x': qqplot_data[0].get_xdata(),
#     'y': qqplot_data[0].get_ydata(),
#     'mode': 'markers',
#     'marker': {
#         'color': '#19d3f3'
#     }
# })

# fig.add_trace({
#     'type': 'scatter',
#     'x': qqplot_data[1].get_xdata(),
#     'y': qqplot_data[1].get_ydata(),
#     'mode': 'lines',
#     'line': {
#         'color': '#636efa'
#     }

# })


# fig['layout'].update({
#     'title': 'Quantile-Quantile Plot',
#     'xaxis': {
#         'title': 'Theoritical Quantities',
#         'zeroline': False
#     },
#     'yaxis': {
#         'title': 'Sample Quantities'
#     },
#     'showlegend': False,
#     'width': 800,
#     'height': 700,
# })
# # fig.show()
# fig.write_image(f"/tmp/QQplot_harmonics_approx_{hn}_{correlation_window[0]}.png")

## Dynamic linear regression